# Live-Cell Paper Renderings

In [ ]:
try:
    import mat73
except ImportError:
    pass

from pathlib import Path
from typing import Sequence

import math
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm


In [ ]:
path = "../../"
path = Path(path).expanduser()
import sys

sys.path.insert(0, str(path))

In [ ]:
import decode
import decode.neuralfitter.inference.functional as infer_func
print(decode.__file__)
log = decode.generic.logging.get_logger(__name__)

%config InlineBackend.figure_format='retina'

In [ ]:
# some paths and hardware settings
# path_dir = "/home/shah/vbc_share/shah/dual_color/results_HMSiRtub_live"
path_dir = "../../results/Fig2c-live_cells"
path_dir = Path(path_dir).expanduser()

path_out = "../../results/Fig2c-live_cells"
path_out = Path(path_out).expanduser()
path_fit = sorted(path_dir.glob("*.h5"))
pattern = "240619_NC_LivingCell_HMSir_Tub"
path_fit = [p for p in path_fit if pattern in p.name][0]

path_fit

In [ ]:
import decode.renderer


z_range = [-400., 400.]
frame_range = [0, 8000]

renderer_frame = decode.renderer.renderer.Renderer2D(
    xextent=(0 * 100., 400*100.),
    yextent=(0 * 100, 400*100.),
    colextent=frame_range,
    px_size=10.,
    sigma_blur=10.,
    rel_clip=0.05,
    contrast=1.5,
    cmap="turbo",
)

renderer_sig = decode.renderer.RendererIndividual2D(
    xextent=(0 * 100., 400*100.),
    yextent=(0 * 100, 400*100.),
    colextent=z_range,
    px_size=10.,
    rel_clip=0.05,
    contrast=1.5,
    cmap="turbo",
    device="cuda:0"
)

renderer_z = decode.renderer.renderer.Renderer2D(
    xextent=(0 * 100., 400*100.),# xextent=(100 * 100., 400*100.),
    yextent=(0 * 100., 400*100.),# yextent=(100. * 100, 400*100.),
    colextent=z_range,
    px_size=10.,
    sigma_blur=10.,
    rel_clip=0.05,
    contrast=1.3,
    cmap="turbo",
)

renderer_mov = decode.renderer.renderer.Renderer2D(
    xextent=(0 * 100., 400*100.),
    yextent=(0 * 100, 400*100.),
    colextent=z_range,
    px_size=10.,
    sigma_blur=10.,
    rel_clip=0.05,
    contrast=3.,
    cmap="turbo",
)

In [ ]:
em = decode.EmitterSet.load(path_fit)

em_rend = em[em.prob > 0.6]
em_rend = em_rend[em_rend.xyz_sig_lat_nm < 40]


In [ ]:

img_z = renderer_z.forward(em_rend, em_rend.xyz[:, 2])
img_frame = renderer_frame.forward(em_rend, em_rend.frame_ix)


# Assuming img_z and img_frame are your images
fig, axs = plt.subplots(figsize=(20, 10), ncols=1)
axs = [axs]

# Create a colormap and normalization
cmap = mpl.cm.turbo

# Loop through each subplot to display the image and add an individual colorbar
for ax, img, title, cmap_range in zip(axs, [img_z, img_frame], ["Z"], [z_range, frame_range]):
    # Display the image
    im = ax.imshow(img)
    ax.set_title(f"Colored by {title} - N: {len(em_rend)}")

    # Create an inset_axes for the colorbar next to each subplot
    # Adjust the [left, bottom, width, height] values as needed for your layout
    cbar_ax = ax.inset_axes([1.05, 0.1, 0.05, 0.8])

    # Create a ScalarMappable with the turbo colormap and normalization
    norm = mpl.colors.Normalize(vmin=cmap_range[0], vmax=cmap_range[1])
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    # Add the colorbar to the inset_axes
    fig.colorbar(sm, cax=cbar_ax)

plt.tight_layout()
# plt.savefig(pout / "overview.png")
# plt.close()

In [ ]:
# define 4 points
x, y = em.px_size.tolist()

xextent=(100 * x, 320 * x)
yextent=(150 * y, 300 * y)

render_zoom = decode.renderer.RendererIndividual2D(
    xextent=xextent,
    yextent=yextent,
    colextent=z_range,
    px_size=10.,
    rel_clip=0.05,
    contrast=1.,
    cmap="turbo",
    batch_size=10000,
    device="cuda:0"
)

img = render_zoom.forward(em_rend, em_rend.xyz[:, 2])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(img.permute(1, 0, -1))

# change ticks
xticks = torch.linspace(0, img.shape[0], 10)
yticks = torch.linspace(0, img.shape[1], 10)

xtick_labels = torch.linspace(xextent[0], xextent[1], 10)
ytick_labels = torch.linspace(yextent[0], yextent[1], 10)

ax.set_xticks(xticks)
ax.set_yticks(yticks)

ax.set_xticklabels([f"{x:.0f}" for x in xtick_labels])
ax.set_yticklabels([f"{y:.0f}" for y in ytick_labels])

plt.show()

In [ ]:
# r = decode.renderer.utils.compute_rectangle_around_line((27000., 27000.), (27000., 33000.), 1000.)
# r

In [ ]:

def compute_rectangle_around_line(p1, p2, w):
    """
    Computes a rectangle around a line defined by two points in 2D.

    Parameters:
    p1 (tuple): Coordinates of the first point (x1, y1).
    p2 (tuple): Coordinates of the second point (x2, y2).
    w (float): Total width of the rectangle.

    Returns:
    tuple: Four points defining the rectangle.
    """
    # Convert points to numpy arrays
    p1 = np.array(p1)
    p2 = np.array(p2)

    # Compute the direction vector of the line
    direction = p2 - p1
    direction = direction / np.linalg.norm(direction)  # Normalize the direction vector

    # Compute the perpendicular vector
    perp_direction = np.array([-direction[1], direction[0]])
    half_width = w / 2

    # Compute the four points of the rectangle
    p1_left = p1 + perp_direction * half_width
    p1_right = p1 - perp_direction * half_width
    p2_left = p2 + perp_direction * half_width
    p2_right = p2 - perp_direction * half_width

    return p1_left, p1_right, p2_left, p2_right

# Example usage
p1 = (0, 0)
p2 = (0, 1)
w = 2
rectangle_points = compute_rectangle_around_line(p1, p2, w)
print("Rectangle Points:", rectangle_points)

In [ ]:
path_fits = sorted(path_dir.glob("*.h5"))
path_fits

In [ ]:
path_fits = sorted(path_dir.glob("*.h5"))


for p in tqdm(path_fits, total=len(path_fits)):
    pout = path_out / p.stem[:-3]
    pout.mkdir(exist_ok=True, parents=False)

    em = decode.EmitterSet.load(p)

    em_rend = em[em.prob > 0.6]
    em_rend = em_rend[em_rend.xyz_sig_lat_nm < 40]

    img_z = renderer_z.forward(em_rend, em_rend.xyz[:, 2])
    img_frame = renderer_frame.forward(em_rend, em_rend.frame_ix)


    # Assuming img_z and img_frame are your images
    fig, axs = plt.subplots(figsize=(20, 10), ncols=2)

    # Create a colormap and normalization
    cmap = mpl.cm.turbo

    # Loop through each subplot to display the image and add an individual colorbar
    for ax, img, title, cmap_range in zip(axs, [img_z, img_frame], ["Z", "Frame"], [z_range, frame_range]):
        # Display the image
        im = ax.imshow(img)
        ax.set_title(f"Colored by {title} - N: {len(em_rend)}")

        # Create an inset_axes for the colorbar next to each subplot
        # Adjust the [left, bottom, width, height] values as needed for your layout
        cbar_ax = ax.inset_axes([1.05, 0.1, 0.05, 0.8])

        # Create a ScalarMappable with the turbo colormap and normalization
        norm = mpl.colors.Normalize(vmin=cmap_range[0], vmax=cmap_range[1])
        sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])

        # Add the colorbar to the inset_axes
        fig.colorbar(sm, cax=cbar_ax)

    plt.tight_layout()
    plt.savefig(pout / "overview.png")
    plt.close()

    # render as movie\
    step_size = 500

    for i in tqdm(range(20)):
        pout_mov = pout / "mov"
        pout_mov.mkdir(exist_ok=True, parents=False)

        f, ax = plt.subplots(figsize=(20, 20))

        ix_low = i * step_size
        ix_high = (i + 1) * step_size
        em_mov = em_rend.iframe[ix_low: ix_high]

        if len(em_mov) < 10000:
            break

        ax.imshow(renderer_mov.forward(em_mov, em_mov.xyz_px[:, 2]))
        ax.set_title(f"Frame: {ix_low} - {ix_high} - N: {len(em_mov)}")

        plt.tight_layout()
        # plt.savefig(pout_mov / f"frame_{i:03d}.png")
        plt.savefig(pout_mov / f"frame_{i:03d}.tiff", format="tiff", dpi=300)
        plt.close()

In [ ]:
# render as movie\
step_size = 500

for i in tqdm(range(20)):
    f, ax = plt.subplots(figsize=(20, 20))

    ix_low = i * step_size
    ix_high = (i + 1) * step_size
    em_mov = em_rend.iframe[ix_low: ix_high]
    ax.imshow(renderer_mov.forward(em_mov, em_mov.xyz_px[:, 2]))
    ax.set_title(f"Frame: {ix_low} - {ix_high}")

    plt.tight_layout()
    plt.savefig(f"/tmp/mov/frame_{i:03d}.png")
    plt.close()